# पाठ ०८ - बहु-एजेन्ट डिजाइन ढाँचा


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## किन मल्टि-एजेण्ट सिस्टमहरू?

वास्तविक-विश्वका कार्यहरू जस्तै यात्रा योजना बनाउन विभिन्न प्रकारका विशेषज्ञता आवश्यक पर्छ — यातायात व्यवस्था, स्थानीय ज्ञान, बजेटिङ, र धेरै केही। सबै कुरामा एकल एजेण्टले प्रयास गर्दा छिट्टै अव्यवस्थित हुन्छ।

मल्टि-एजेण्ट सिस्टमहरू यसलाई **विशेषज्ञता** मार्फत समाधान गर्छन्: प्रत्येक एजेण्टले एक क्षेत्रको विशेषज्ञतामा केन्द्रित हुन्छ, जसले सामान्य एजेण्टभन्दा उच्च गुणस्तरको परिणाम दिन्छ। यसले **स्केलेबिलिटी** पनि सुधार गर्छ — नयाँ एजेण्टहरू (जस्तै, उडान विशेषज्ञ, रेस्टुरेन्ट समीक्षक) थप्न सकिन्छ बिना अवस्थित कार्यप्रवाह पुनःलेखनको। एजेण्टहरू संरचित पाइपलाइनमार्फत सँगै मिलेर काम गर्छन्, एकबाट अर्कोप्रति प्रसङ्ग पास गर्दै।


## विशेष एजेन्टहरू सिर्जना गर्दै


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## एक उत्तरोत्तर कार्यप्रवाह निर्माण गर्दै

`WorkflowBuilder` ले तपाईंलाई एजेन्टहरूलाई निर्देशित ग्राफमा जडान गर्न अनुमति दिन्छ। यहाँ हामी एउटा साधारण दुई-चरण पाइपलाइन बनाउँछौं: **TravelPlanner** यात्रा योजना तयार गर्छ, त्यसपछि **TravelConcierge** यसलाई समीक्षा गरेर सुधार गर्दछ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ्लोमा थप एजेन्टहरू थप्दै

मल्टि-एजेन्ट प्याटर्नको एउटा सबैभन्दा ठूलो फाइदा यसको विस्तार गर्न कति सजिलो छ भन्ने हो। तल हामीले एउटा **BudgetReviewer** एजेन्ट थपेका छौं जसले यात्रुको बजेटसँग योजना जाँच्छ, लागत सीमा नाघ्न सक्ने वस्तुहरूलाई झण्डा लगाउँछ, र पैसा बचत गर्न सक्ने विकल्पहरू सुझाउँछ। वर्कफ्लो अहिले तीन एजेन्टहरूलाई क्रमशः चलाउँछ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. **विशेषीकृत एजेन्टहरू सिर्जना गर्ने** — प्रत्येकको केन्द्रित भूमिका (योजना बनाउने, कन्सियर्ज, बजेट समीक्षा)।
2. **एजेन्टहरूलाई क्रमिक कार्यप्रवाहमा जोड्ने** `WorkflowBuilder` र `add_edge` प्रयोग गरेर।
3. **बहु-एजेन्ट पाइपलाइनबाट आउटपुट स्ट्रिम गर्ने**, कुन एजेन्ट बोलिरहेको छ ट्र्याक गर्दै।
4. **कार्यप्रवाह विस्तार गर्ने** श्रृंखलामा नयाँ एजेन्टहरू थपेर बिना पहिलेका एजेन्टहरूलाई संशोधन नगरी।

बहु-एजेन्ट डिजाइन नमूनाले प्रत्येक एजेन्टलाई सरल राख्छ जब कि एक्लो एजेन्टले गर्न नसक्ने समृद्ध, थप गहिराइसित समीक्षा गरिएका परिणामहरू उत्पादन गर्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**उत्तरदायित्व अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) को प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयासरत छौं भने पनि, कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटिहरू वा गलतताहरू हुनसक्छन्। मूल दस्तावेज़ यसको मातृ भाषामा नै अधिकारिक स्रोत मानिनेछ। महत्वपूर्ण जानकारीको लागि, पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार हुने छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
